# Expense Claim Crew: Domain Expert AI Agents for Expense Reimbursement
### Aman Aggarwal (065064) | PGDM (BDA) 2025-27 | FORE School of Management

**Business process:** employee expense reimbursement. Today a finance executive checks every claim by hand: reads the receipt, looks up the policy limit for the employee's grade, checks dates, and watches for duplicates or split bills. It is slow, repetitive and inconsistent between reviewers.

**Solution:** a crew of four domain expert AI agents (CrewAI + Google Gemini) that audit each claim and recommend **APPROVE**, **FLAG** (manager review) or **REJECT**, with reasons and a message to the employee.

| Agent | Expertise | Tool it uses |
|---|---|---|
| 1. Receipt Analyst | Reads messy receipt text: vendor, date, total, non-reimbursable items such as alcohol or fines | Get claim details |
| 2. Expense Policy Expert | Applies grade-wise limits, receipt and timeliness rules | Check policy rules |
| 3. Fraud and Risk Auditor | Finds duplicates, split bills, weekend client meals, receipt mismatches | Run fraud checks |
| 4. Finance Approver | Combines all findings into the final decision and writes the employee message | (none: reasoning only) |

**Design principle:** the LLM agents read, interpret and explain; every number (limits, day counts, duplicate matches) comes from deterministic Python tools. During the setup test, a single agent with no tools invented a fact ("two attendees") to approve an over-limit dinner. Tools prevent that kind of guess.

**Data:** a fictional company (Nimbus Analytics Pvt. Ltd.), its expense policy, 10 employees, 4 past claims and 16 new claims with known correct answers, so the crew's accuracy can be measured.

## 1. Setup

In [ ]:
import os, json
from dotenv import load_dotenv

load_dotenv()
os.environ["CREWAI_TRACING_ENABLED"] = "false"   # no tracing prompt
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))

## 2. Sample data

Running these cells writes the data files into the `data` folder. Edit them here if you want to change the policy or add claims.

### 2.1 Company expense policy

In [ ]:
%%writefile data/expense_policy.md
# Nimbus Analytics Pvt. Ltd.: Employee Expense Reimbursement Policy (v2.1, effective 1 April 2026)

Nimbus Analytics is a fictional company used for this project. All amounts are in Indian Rupees (INR).

## 1. Employee grades
- G1: Associates and Analysts
- G2: Managers
- G3: Directors and above

## 2. Category limits (maximum reimbursable amount)

| Category | Unit | G1 | G2 | G3 |
|---|---|---|---|---|
| Client Meals | per person, per meal | 1,000 | 1,500 | 2,500 |
| Local Travel | per day | 1,000 | 1,500 | 2,500 |
| Hotel (Tier 1 city) | per night | 4,000 | 6,000 | 9,000 |
| Hotel (Tier 2 city) | per night | 3,000 | 4,500 | 7,000 |
| Airfare (economy) | per one-way trip | 8,000 | 12,000 | 20,000 |
| Office Supplies | per claim | 2,000 | 2,000 | 2,000 |
| Internet and Mobile | per month | 1,000 | 1,000 | 1,000 |

Tier 1 cities: Delhi, Mumbai, Bengaluru, Chennai, Kolkata, Hyderabad, Pune. All other cities are Tier 2.
For Client Meals, the number of attendees (including the employee) must be stated on the claim.

## 3. General rules
- R1 Receipts: an original receipt is required for every claim above INR 500. Claims above INR 500 without a receipt are rejected and may be resubmitted with a receipt.
- R2 Timeliness: claims must be submitted within 30 days of the expense date. Late claims are rejected.
- R3 Non-reimbursable items: alcohol, traffic or parking fines, penalties, personal items, spa and salon services. A claim made up entirely of non-reimbursable items is rejected. If a receipt contains some non-reimbursable items, only the eligible amount can be approved, and the claim is flagged for review.
- R4 Limits: a claim above its category limit is flagged for manager review. The recommended amount is capped at the limit.
- R5 Duplicates: a claim matching an earlier claim (same employee, vendor, date and amount) is rejected as a duplicate.
- R6 Receipt match: the claimed amount and date must match the receipt. Any mismatch is flagged, and the recommended amount is the lower of the claimed and receipt amounts.
- R7 Unusual patterns are flagged for review: client meals on a Saturday or Sunday, several claims from the same vendor on the same day that together exceed a limit (split bills), and round amounts of INR 5,000 or more.

## 4. Decisions
- REJECT if any of R1, R2, R5 applies, or the whole claim is non-reimbursable (R3).
- Otherwise FLAG (manager review) if any of R3 (partial), R4, R6 or R7 applies.
- Otherwise APPROVE.

### 2.2 Employees (grade decides the limits)

In [ ]:
%%writefile data/employees.csv
employee_id,name,grade,department,manager
E01,Riya Mehta,G1,Analytics,Arjun Rao
E02,Karan Sethi,G2,Sales,Neha Kapoor
E03,Priya Nair,G1,Consulting,Arjun Rao
E04,Vikram Joshi,G2,Operations,Neha Kapoor
E05,Ananya Das,G1,Finance,Sanjay Iyer
E06,Rohan Malhotra,G3,Sales,Managing Director
E07,Sneha Kulkarni,G2,Consulting,Sanjay Iyer
E08,Aditya Verma,G1,Sales,Karan Sethi
E09,Meera Pillai,G3,Consulting,Managing Director
E10,Farhan Sheikh,G1,Analytics,Arjun Rao

### 2.3 Past reimbursed claims (used to catch duplicates)

In [ ]:
%%writefile data/claims_history.csv
claim_id,employee_id,expense_date,category,vendor,amount,status
H101,E07,2026-09-18,Client Meals,Spice Route Restaurant,2640,Reimbursed
H102,E02,2026-09-10,Local Travel,QuickRide Cabs,1180,Reimbursed
H103,E06,2026-09-05,Airfare,SkyWays Airlines,17200,Reimbursed
H104,E01,2026-08-28,Office Supplies,PaperPoint Stationers,640,Reimbursed

### 2.4 New claims to audit

`units` means attendees for client meals, nights for hotels, days for local travel, and 1 otherwise. `receipt_text` is the text read from the receipt (as an OCR scan would give it).

In [ ]:
%%writefile data/claims.csv
claim_id,employee_id,expense_date,submitted_on,category,vendor,city,units,amount,receipt_text
C001,E01,2026-09-24,2026-09-29,Client Meals,Spice Route Restaurant,Delhi,2,1850,"SPICE ROUTE RESTAURANT | Connaught Place, New Delhi | Date: 24/09/2026 | Table 7, Covers: 2 | Paneer Tikka 420 | Dal Makhani 380 | Butter Naan x4 240 | Veg Biryani 450 | Fresh Lime Soda x2 180 | Sub-total 1670 | GST 5% 84 | Service charge 96 | TOTAL INR 1850"
C002,E02,2026-09-30,2026-10-02,Local Travel,QuickRide Cabs,Mumbai,1,1320,QUICKRIDE CABS | Trip receipts for 30 Sep 2026 | Andheri to BKC 410 | BKC to Lower Parel 365 | Lower Parel to Andheri 545 | Total fare INR 1320 | Paid via UPI
C003,E03,2026-09-15,2026-09-20,Hotel,Hotel Lotus Residency,Bengaluru,2,9600,"HOTEL LOTUS RESIDENCY, Bengaluru | Guest: Priya Nair | Check-in 15/09/2026 Check-out 17/09/2026 | Deluxe Room 2 nights x 4,285.71 = 8571.43 | GST 12% 1028.57 | GRAND TOTAL INR 9600.00"
C004,E04,2026-09-26,2026-10-01,Client Meals,The Grill House,Pune,3,5600,"THE GRILL HOUSE, Koregaon Park, Pune | 26-Sep-2026 | Covers 3 | Grilled Chicken Platter 1350 | Paneer Steak 890 | Pasta Arrabiata 720 | Caesar Salad 540 | Desserts x3 600 | Kingfisher Beer x4 1200 | Sub-total 5300 | GST 300 | TOTAL INR 5600"
C005,E05,2026-10-03,2026-10-05,Office Supplies,PaperPoint Stationers,Kolkata,1,780,NO RECEIPT ATTACHED
C006,E06,2026-09-22,2026-09-25,Airfare,SkyWays Airlines,Delhi,1,18400,"SKYWAYS AIRLINES E-TICKET | Passenger: Rohan Malhotra | DEL to BOM | 22 Sep 2026 | Class: Economy (Flex) | Base fare 15,900 | Taxes and fees 2,500 | TOTAL INR 18,400"
C007,E07,2026-09-18,2026-10-04,Client Meals,Spice Route Restaurant,Delhi,2,2640,"SPICE ROUTE RESTAURANT | Connaught Place, New Delhi | Date: 18/09/2026 | Covers: 2 | Chicken Tikka 520 | Rogan Josh 640 | Garlic Naan x4 280 | Jeera Rice 320 | Gulab Jamun x2 240 | Mocktails x2 400 | Sub-total 2400 | GST 120 | Service charge 120 | TOTAL INR 2640"
C008,E07,2026-08-20,2026-10-05,Local Travel,QuickRide Cabs,Pune,1,1450,QUICKRIDE CABS | Trip receipts for 20 Aug 2026 | Hinjewadi to Shivajinagar 780 | Shivajinagar to Hinjewadi 670 | Total fare INR 1450
C009,E02,2026-10-01,2026-10-03,Local Travel,Mumbai Traffic Police,Mumbai,1,2000,E-CHALLAN | Mumbai Traffic Police | Date 01/10/2026 | Vehicle MH02AB1234 | Offence: Signal jumping | Fine amount INR 2000 | Status: Paid
C010,E08,2026-10-04,2026-10-06,Client Meals,Coastal Kitchen,Chennai,2,2000,"COASTAL KITCHEN, Besant Nagar, Chennai | Sunday 04/10/2026 | Covers 2 | Fish Curry Meals x2 760 | Prawn Fry 520 | Appam x4 240 | Filter Coffee x2 120 | Sub-total 1640 | GST 82 | Service 278 | TOTAL INR 2000"
C011,E04,2026-10-07,2026-10-08,Office Supplies,OfficeMart Supplies,Pune,1,1900,"OFFICEMART SUPPLIES, Pune | Invoice 4471 | 07/10/2026 | A4 paper reams x10 1450 | Whiteboard markers x10 450 | TOTAL INR 1900"
C012,E04,2026-10-07,2026-10-08,Office Supplies,OfficeMart Supplies,Pune,1,1850,"OFFICEMART SUPPLIES, Pune | Invoice 4472 | 07/10/2026 | Printer toner cartridge x1 1850 | TOTAL INR 1850"
C013,E09,2026-09-28,2026-10-01,Hotel,Hotel Ganga View,Jaipur,1,6500,"HOTEL GANGA VIEW, Jaipur | Guest: Meera Pillai | Check-in 28/09/2026 Check-out 29/09/2026 | Executive Room 1 night 5803.57 | GST 12% 696.43 | TOTAL INR 6500.00"
C014,E10,2026-09-30,2026-10-02,Internet and Mobile,AirLink Broadband,Hyderabad,1,999,AIRLINK BROADBAND | Monthly bill September 2026 | Plan: Fibre 100 Mbps | Rental 847 | GST 18% 152 | Amount due INR 999
C015,E03,2026-10-02,2026-10-05,Local Travel,QuickRide Cabs,Bengaluru,1,1200,QUICKRIDE CABS | Trip receipts for 02 Oct 2026 | Koramangala to Whitefield 520 | Whitefield to Koramangala 430 | Total fare INR 950
C016,E06,2026-10-06,2026-10-08,Client Meals,Saffron Terrace,Mumbai,4,9200,"SAFFRON TERRACE, Worli, Mumbai | 06/10/2026 | Covers 4 | Tasting menu x4 7600 | Mocktails x4 800 | Sub-total 8400 | GST 420 | Service 380 | TOTAL INR 9200"

### 2.5 Expected outcomes (the answer key, hand-checked against the policy)

In [ ]:
%%writefile data/expected_outcomes.csv
claim_id,expected_decision,expected_amount,reason
C001,APPROVE,1850,"Within limit (2 x 1,000), receipt matches"
C002,APPROVE,1320,"Within daily limit of 1,500"
C003,FLAG,8000,"Hotel above Tier 1 G1 limit (2 x 4,000)"
C004,FLAG,4400,"Beer of 1,200 removed (alcohol), eligible 4,400 within 3 x 1,500; also a Saturday client meal"
C005,REJECT,0,No receipt for a claim above 500
C006,APPROVE,18400,Economy airfare within G3 limit
C007,REJECT,0,Duplicate of reimbursed claim H101
C008,REJECT,0,Submitted 46 days after expense (limit 30)
C009,REJECT,0,Traffic fine is non-reimbursable
C010,FLAG,2000,Client meal on a Sunday
C011,FLAG,1900,"Split bill: two same-vendor claims on one day total 3,750, above 2,000"
C012,FLAG,1850,"Split bill with C011 (3,750 together, above 2,000)"
C013,APPROVE,6500,"Within Tier 2 G3 hotel limit of 7,000"
C014,APPROVE,999,"Within monthly limit of 1,000"
C015,FLAG,950,"Claimed 1,200 but receipt shows 950"
C016,APPROVE,9200,"Within limit (4 x 2,500)"

## 3. The crew: tools, agents and tasks

All the crew logic lives in `expense_crew.py` so the Streamlit app can reuse exactly the same agents later. Running this cell writes the file.

In [ ]:
%%writefile expense_crew.py
"""
expense_crew.py
Expense Claim Crew: domain expert AI agents (CrewAI + Gemini) that audit employee
expense claims against company policy.
Author: Aman Aggarwal (065064), FORE School of Management

Design principle: the LLM agents read messy receipts, interpret findings and write
explanations; every calculation (limits, dates, duplicates) is done by deterministic
Python tools, so the agents never have to guess numbers.
"""
import json
import os
import re
import time
from datetime import date
from typing import List, Literal

os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")   # no tracing prompt
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
# In CrewAI 1.9 this flag only switches off the interactive "view your traces? [y/N]"
# prompt, which would otherwise pause every run for 20 s (and hang a web app).
os.environ.setdefault("CREWAI_TESTING", "true")

import pandas as pd
from pydantic import BaseModel, Field
from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool

# ------------------------------------------------------------------
# Compatibility fix: Gemini rejects requests whose last message is from the model
# ("400 Requests ending with a model turn are not supported"). CrewAI's tool loop
# can end on the agent's own "Observation" message, so we add a short user turn.
# ------------------------------------------------------------------
GEMINI_RPM = int(os.getenv("GEMINI_RPM", "10"))   # stay safely under the free-tier limit
_last_call = [0.0]


def _wait_for_slot():
    gap = 60.0 / GEMINI_RPM
    wait = _last_call[0] + gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()


def _patch_gemini_last_turn():
    try:
        from google.genai import types as gtypes
        from crewai.llms.providers.gemini.completion import GeminiCompletion
    except ImportError:
        return
    if getattr(GeminiCompletion, "_last_turn_patched", False):
        return
    original = GeminiCompletion._format_messages_for_gemini

    def patched(self, messages):
        contents, system_instruction = original(self, messages)
        if contents and contents[-1].role == "model":
            parts = contents[-1].parts or []
            if not any(getattr(p, "function_call", None) for p in parts):
                contents.append(gtypes.Content(
                    role="user", parts=[gtypes.Part.from_text(text="Continue.")]))
        return contents, system_instruction

    GeminiCompletion._format_messages_for_gemini = patched

    # Free-tier pacing: never send more than GEMINI_RPM requests per minute (all agents
    # together), and when Google answers 429 (quota) or 503 (busy), wait the time it asks
    # for and try the same call again instead of failing the whole claim.
    original_call = GeminiCompletion.call

    def paced_call(self, *args, **kwargs):
        for attempt in range(5):
            _wait_for_slot()
            try:
                return original_call(self, *args, **kwargs)
            except Exception as e:
                msg = str(e)
                if "PerDay" in msg or "per day" in msg.lower():
                    raise  # daily quota used up: waiting a minute will not help
                if not any(code in msg for code in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")):
                    raise
                if attempt == 4:
                    raise
                m = re.search(r"retry in ([0-9.]+)s", msg)
                wait = float(m.group(1)) + 3 if m else 30
                print(f"   Gemini busy or rate-limited, waiting {wait:.0f} s then retrying ...")
                time.sleep(min(wait, 70))

    GeminiCompletion.call = paced_call
    GeminiCompletion._last_turn_patched = True


_patch_gemini_last_turn()

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")

# ------------------------------------------------------------------
# 1. Data
# ------------------------------------------------------------------
CLAIMS = pd.read_csv(os.path.join(DATA_DIR, "claims.csv"), dtype={"claim_id": str})
EMPLOYEES = pd.read_csv(os.path.join(DATA_DIR, "employees.csv"))
HISTORY = pd.read_csv(os.path.join(DATA_DIR, "claims_history.csv"))
with open(os.path.join(DATA_DIR, "expense_policy.md")) as f:
    POLICY_TEXT = f.read()

NO_RECEIPT = "NO RECEIPT ATTACHED"
TIER1 = {"Delhi", "Mumbai", "Bengaluru", "Chennai", "Kolkata", "Hyderabad", "Pune"}
LIMITS = {  # category: (unit, {grade: limit per unit})
    "Client Meals": ("person", {"G1": 1000, "G2": 1500, "G3": 2500}),
    "Local Travel": ("day", {"G1": 1000, "G2": 1500, "G3": 2500}),
    "Hotel Tier 1": ("night", {"G1": 4000, "G2": 6000, "G3": 9000}),
    "Hotel Tier 2": ("night", {"G1": 3000, "G2": 4500, "G3": 7000}),
    "Airfare": ("one-way trip", {"G1": 8000, "G2": 12000, "G3": 20000}),
    "Office Supplies": ("claim", {"G1": 2000, "G2": 2000, "G3": 2000}),
    "Internet and Mobile": ("month", {"G1": 1000, "G2": 1000, "G3": 1000}),
}
SUBMISSION_DAYS = 30
RECEIPT_THRESHOLD = 500


def _claim(claim_id: str) -> dict:
    row = CLAIMS[CLAIMS.claim_id == claim_id.strip().upper()]
    if row.empty:
        raise ValueError(f"Claim {claim_id} not found")
    return row.iloc[0].to_dict()


def _employee(employee_id: str) -> dict:
    return EMPLOYEES[EMPLOYEES.employee_id == employee_id].iloc[0].to_dict()


def _limit(claim: dict, grade: str):
    category = claim["category"]
    if category == "Hotel":
        category = "Hotel Tier 1" if claim["city"] in TIER1 else "Hotel Tier 2"
    unit, by_grade = LIMITS[category]
    per_unit = by_grade[grade]
    return category, unit, per_unit, per_unit * int(claim["units"])


# ------------------------------------------------------------------
# 2. Tools (deterministic Python the agents call)
# ------------------------------------------------------------------
@tool("Get claim details")
def get_claim_details(claim_id: str) -> str:
    """Return the expense claim form, the receipt text and the employee's grade
    for one claim ID such as C001."""
    c = _claim(claim_id)
    e = _employee(c["employee_id"])
    return json.dumps({**c, "employee_name": e["name"], "grade": e["grade"],
                       "department": e["department"]}, default=str)


@tool("Check policy rules")
def check_policy_rules(claim_id: str, receipt_total: float,
                       non_reimbursable_amount: float) -> str:
    """Apply the company expense policy (limits, receipts, timeliness, non-reimbursable
    items) to a claim. Pass the receipt total and the total of non-reimbursable items
    found on the receipt (0 if none). Returns triggered rules and the eligible amount."""
    c = _claim(claim_id)
    grade = _employee(c["employee_id"])["grade"]
    amount = float(c["amount"])
    category, unit, per_unit, max_allowed = _limit(c, grade)
    triggers = []

    has_receipt = c["receipt_text"].strip().upper() != NO_RECEIPT
    if not has_receipt and amount > RECEIPT_THRESHOLD:
        triggers.append({"rule": "R1 Receipts", "effect": "REJECT",
                         "detail": f"No receipt for a claim of INR {amount:,.0f} (above {RECEIPT_THRESHOLD})."})

    days = (date.fromisoformat(c["submitted_on"]) - date.fromisoformat(c["expense_date"])).days
    if days > SUBMISSION_DAYS:
        triggers.append({"rule": "R2 Timeliness", "effect": "REJECT",
                         "detail": f"Submitted {days} days after the expense (limit {SUBMISSION_DAYS})."})

    base = min(amount, float(receipt_total)) if has_receipt and receipt_total else amount
    eligible = max(base - float(non_reimbursable_amount or 0), 0.0)
    if non_reimbursable_amount and non_reimbursable_amount > 0:
        if eligible <= 0:
            triggers.append({"rule": "R3 Non-reimbursable", "effect": "REJECT",
                             "detail": "The whole claim is for non-reimbursable items."})
        else:
            triggers.append({"rule": "R3 Non-reimbursable", "effect": "FLAG",
                             "detail": f"INR {non_reimbursable_amount:,.0f} of non-reimbursable items removed."})

    if eligible > max_allowed:
        triggers.append({"rule": "R4 Limit", "effect": "FLAG",
                         "detail": f"Eligible INR {eligible:,.0f} is above the {category} limit of "
                                   f"INR {max_allowed:,.0f} ({c['units']} x {per_unit:,} per {unit}, grade {grade})."})

    return json.dumps({
        "claim_id": c["claim_id"], "category": category, "grade": grade,
        "limit_per_unit": per_unit, "unit": unit, "units": int(c["units"]),
        "max_allowed_amount": max_allowed, "claimed_amount": amount,
        "eligible_amount": round(eligible, 2),
        "recommended_amount": round(min(eligible, max_allowed), 2),
        "days_to_submit": days, "rule_triggers": triggers,
    })


@tool("Run fraud checks")
def run_fraud_checks(claim_id: str, receipt_date: str, receipt_total: float) -> str:
    """Check a claim for duplicates, split bills, weekend client meals, suspicious round
    amounts and mismatches between the claim and the receipt. Pass the receipt date as
    YYYY-MM-DD (empty if no receipt) and the receipt total (0 if no receipt)."""
    c = _claim(claim_id)
    grade = _employee(c["employee_id"])["grade"]
    amount = float(c["amount"])
    triggers = []

    same = lambda df: df[(df.employee_id == c["employee_id"]) & (df.vendor == c["vendor"])
                         & (df.expense_date == c["expense_date"])]
    dup_hist = same(HISTORY)[lambda d: d.amount == amount]
    dup_now = same(CLAIMS)[lambda d: (d.amount == amount) & (d.claim_id < c["claim_id"])]
    if len(dup_hist) or len(dup_now):
        ref = (list(dup_hist.claim_id) + list(dup_now.claim_id))[0]
        triggers.append({"rule": "R5 Duplicate", "effect": "REJECT",
                         "detail": f"Same employee, vendor, date and amount as claim {ref}."})

    siblings = same(CLAIMS)[lambda d: (d.claim_id != c["claim_id"]) & (d.amount != amount)]
    if len(siblings):
        _, _, _, max_allowed = _limit(c, grade)
        combined = amount + siblings.amount.sum()
        if combined > max_allowed:
            triggers.append({"rule": "R7 Split bill", "effect": "FLAG",
                             "detail": f"With {', '.join(siblings.claim_id)} the same-day total from "
                                       f"{c['vendor']} is INR {combined:,.0f}, above the limit of INR {max_allowed:,.0f}."})

    weekday = date.fromisoformat(c["expense_date"]).strftime("%A")
    if c["category"] == "Client Meals" and weekday in ("Saturday", "Sunday"):
        triggers.append({"rule": "R7 Weekend client meal", "effect": "FLAG",
                         "detail": f"Client meal on a {weekday}; business purpose needs confirmation."})

    if amount >= 5000 and amount % 1000 == 0:
        triggers.append({"rule": "R7 Round amount", "effect": "FLAG",
                         "detail": f"Round amount of INR {amount:,.0f}."})

    if receipt_total and abs(float(receipt_total) - amount) > 1:
        triggers.append({"rule": "R6 Receipt match", "effect": "FLAG",
                         "detail": f"Claimed INR {amount:,.0f} but the receipt shows INR {float(receipt_total):,.0f}."})
    if receipt_date and receipt_date != c["expense_date"]:
        triggers.append({"rule": "R6 Receipt match", "effect": "FLAG",
                         "detail": f"Claim date {c['expense_date']} but receipt date {receipt_date}."})

    return json.dumps({"claim_id": c["claim_id"], "expense_weekday": weekday,
                       "rule_triggers": triggers})


# ------------------------------------------------------------------
# 3. Structured outputs (what each agent must return)
# ------------------------------------------------------------------
class RuleTrigger(BaseModel):
    rule: str
    effect: Literal["REJECT", "FLAG"]
    detail: str


class NonReimbursableItem(BaseModel):
    item: str
    amount: float
    reason: str


class ReceiptExtraction(BaseModel):
    receipt_present: bool
    vendor: str = ""
    receipt_date: str = Field("", description="YYYY-MM-DD, empty if no receipt")
    receipt_total: float = Field(0, description="Final total on the receipt in INR, 0 if no receipt")
    non_reimbursable_items: List[NonReimbursableItem] = []
    notes: str = ""


class PolicyReport(BaseModel):
    rule_triggers: List[RuleTrigger] = []
    max_allowed_amount: float
    eligible_amount: float
    recommended_amount: float
    summary: str


class FraudReport(BaseModel):
    rule_triggers: List[RuleTrigger] = []
    risk_level: Literal["Low", "Medium", "High"]
    summary: str


class FinalDecision(BaseModel):
    decision: Literal["APPROVE", "FLAG", "REJECT"]
    recommended_amount: float
    reasons: List[str]
    employee_message: str


# ------------------------------------------------------------------
# 4. Agents and tasks
# ------------------------------------------------------------------
def make_llm(model: str = "gemini/gemini-flash-lite-latest", api_key: str | None = None) -> LLM:
    return LLM(model=model, api_key=api_key or os.getenv("GEMINI_API_KEY"), temperature=0.1)


def build_crew(llm, verbose: bool = False, max_rpm: int | None = None,
               task_callback=None) -> Crew:
    receipt_agent = Agent(
        role="Receipt Analyst",
        goal="Read the receipt for claim {claim_id} accurately and extract its facts",
        backstory=("You have processed thousands of Indian restaurant bills, cab receipts, hotel "
                   "invoices and e-tickets. You read messy receipt text, find the final total and "
                   "date, and spot items a company never pays for: alcohol (beer, wine, whisky, "
                   "spirits), traffic or parking fines, penalties, personal items, spa and salon. "
                   "You never guess: if something is not on the receipt, you say so."),
        tools=[get_claim_details], llm=llm, verbose=verbose, allow_delegation=False)

    policy_agent = Agent(
        role="Expense Policy Expert",
        goal="Decide which company policy rules apply to claim {claim_id}",
        backstory=("You are the company's expense policy owner in Finance. You always use the "
                   "'Check policy rules' tool for limits, dates and amounts, because policy "
                   "decisions must be exact. You know this policy by heart:\n\n" + POLICY_TEXT),
        tools=[check_policy_rules], llm=llm, verbose=verbose, allow_delegation=False)

    fraud_agent = Agent(
        role="Fraud and Risk Auditor",
        goal="Detect duplicate, split, suspicious or mismatched claims for claim {claim_id}",
        backstory=("You are an internal auditor trained in expense fraud patterns: duplicate "
                   "submissions, bills split to stay under limits, weekend entertainment and "
                   "receipts that do not match the claim. You rely on the 'Run fraud checks' tool "
                   "for evidence and never accuse without it."),
        tools=[run_fraud_checks], llm=llm, verbose=verbose, allow_delegation=False)

    approver_agent = Agent(
        role="Finance Approver",
        goal="Make a fair, policy-compliant final decision on claim {claim_id} and explain it",
        backstory=("You are the Finance Controller who signs off reimbursements. You combine the "
                   "receipt, policy and fraud findings, apply the decision rules exactly, and write "
                   "a short, polite message the employee will understand."),
        llm=llm, verbose=verbose, allow_delegation=False)

    receipt_task = Task(
        description=("Use the 'Get claim details' tool for claim {claim_id}. Read the receipt_text. "
                     "If it says 'NO RECEIPT ATTACHED', set receipt_present to false and the total to 0. "
                     "Otherwise extract the vendor, the receipt date as YYYY-MM-DD and the final total "
                     "in INR, and list every non-reimbursable item with its amount."),
        expected_output="The receipt facts in the required structure.",
        agent=receipt_agent, output_pydantic=ReceiptExtraction)

    policy_task = Task(
        description=("For claim {claim_id}, call the 'Check policy rules' tool with the receipt total "
                     "and the sum of non-reimbursable item amounts from the Receipt Analyst. Report "
                     "exactly the rule triggers and amounts the tool returns, then summarise in one "
                     "or two sentences."),
        expected_output="The policy findings in the required structure.",
        agent=policy_agent, context=[receipt_task], output_pydantic=PolicyReport)

    fraud_task = Task(
        description=("For claim {claim_id}, call the 'Run fraud checks' tool with the receipt date and "
                     "receipt total from the Receipt Analyst. Report exactly the rule triggers it returns. "
                     "Risk level: High if any REJECT trigger, Medium if any FLAG trigger, else Low."),
        expected_output="The fraud findings in the required structure.",
        agent=fraud_agent, context=[receipt_task], output_pydantic=FraudReport)

    decision_task = Task(
        description=("Make the final decision on claim {claim_id} using the policy and fraud findings.\n"
                     "Decision rules: REJECT if any trigger has effect REJECT (recommended amount 0). "
                     "Otherwise FLAG if any trigger has effect FLAG (recommended amount = the policy "
                     "report's recommended amount, or the receipt total if lower when the receipt does "
                     "not match). Otherwise APPROVE the policy report's recommended amount.\n"
                     "Give one reason per triggered rule (or 'All policy checks passed') and a two or "
                     "three sentence message to the employee."),
        expected_output="The final decision in the required structure.",
        agent=approver_agent, context=[receipt_task, policy_task, fraud_task],
        output_pydantic=FinalDecision)

    return Crew(agents=[receipt_agent, policy_agent, fraud_agent, approver_agent],
                tasks=[receipt_task, policy_task, fraud_task, decision_task],
                process=Process.sequential, verbose=verbose, max_rpm=max_rpm,
                tracing=False, task_callback=task_callback)


# ------------------------------------------------------------------
# 5. Running claims (with retry and model fallback)
# ------------------------------------------------------------------
# Flash-Lite first: on the free tier it is busy (503) far less often than Flash
FALLBACK_MODELS = ["gemini/gemini-flash-lite-latest", "gemini/gemini-flash-latest"]


def run_claim(claim_id: str, models=FALLBACK_MODELS, verbose: bool = False,
              api_key: str | None = None, retries: int = 2, task_callback=None) -> dict:
    """Run the four-agent crew on one claim. Returns each agent's structured output."""
    last_error = None
    for model in models:
        for attempt in range(retries):
            try:
                crew = build_crew(make_llm(model, api_key), verbose=verbose,
                                  task_callback=task_callback)
                start = time.time()
                result = crew.kickoff(inputs={"claim_id": claim_id})
                outs = [t.pydantic.model_dump() if t.pydantic else {"raw": t.raw}
                        for t in result.tasks_output]
                return {"claim_id": claim_id, "model": model,
                        "seconds": round(time.time() - start, 1),
                        "receipt": outs[0], "policy": outs[1], "fraud": outs[2], "decision": outs[3]}
            except Exception as e:  # busy model, rate limit or bad output
                last_error = e
                if "PerDay" in str(e) or "per day" in str(e).lower():
                    return {"claim_id": claim_id, "error": "DAILY QUOTA REACHED: " + str(e)[:300]}
                busy = any(s in str(e) for s in ("503", "429", "UNAVAILABLE", "RESOURCE_EXHAUSTED"))
                time.sleep(20 if busy else 3)
    return {"claim_id": claim_id, "error": str(last_error)[:500]}


# ------------------------------------------------------------------
# 6. Adding a new claim (used by the Streamlit app)
# ------------------------------------------------------------------
CATEGORIES = ["Client Meals", "Local Travel", "Hotel", "Airfare", "Office Supplies",
              "Internet and Mobile"]


def add_claim(employee_id: str, expense_date: str, submitted_on: str, category: str,
              vendor: str, city: str, units: int, amount: float, receipt_text: str) -> str:
    """Add a new claim to the in-memory claims table and return its new ID."""
    global CLAIMS
    if employee_id not in set(EMPLOYEES.employee_id):
        raise ValueError("Unknown employee")
    if category not in CATEGORIES:
        raise ValueError("Unknown category")
    new_id = f"N{int(time.time() * 1000) % 10_000_000:07d}"
    row = {"claim_id": new_id, "employee_id": employee_id, "expense_date": expense_date,
           "submitted_on": submitted_on, "category": category, "vendor": vendor.strip(),
           "city": city.strip(), "units": int(units), "amount": float(amount),
           "receipt_text": receipt_text.strip() or NO_RECEIPT}
    CLAIMS = pd.concat([CLAIMS, pd.DataFrame([row])], ignore_index=True)
    return new_id

In [ ]:
import importlib, pandas as pd
import expense_crew as ec
importlib.reload(ec)

pd.set_option("display.max_colwidth", 80)
ec.CLAIMS[["claim_id", "employee_id", "expense_date", "category", "vendor", "city", "units", "amount"]]

## 4. Sanity check of the tools (no AI, no API calls)

Before involving any AI, we feed the tools the receipt facts a perfect reader would extract and apply the policy's decision rules in plain Python. If the tools are right, this matches the answer key 16 out of 16. It also gives a **rules-only baseline** to compare the agents against.

In [ ]:
perfect_receipts = {  # claim: (receipt date, receipt total, non-reimbursable amount)
    "C001": ("2026-09-24", 1850, 0),    "C002": ("2026-09-30", 1320, 0),
    "C003": ("2026-09-15", 9600, 0),    "C004": ("2026-09-26", 5600, 1200),
    "C005": ("", 0, 0),                 "C006": ("2026-09-22", 18400, 0),
    "C007": ("2026-09-18", 2640, 0),    "C008": ("2026-08-20", 1450, 0),
    "C009": ("2026-10-01", 2000, 2000), "C010": ("2026-10-04", 2000, 0),
    "C011": ("2026-10-07", 1900, 0),    "C012": ("2026-10-07", 1850, 0),
    "C013": ("2026-09-28", 6500, 0),    "C014": ("2026-09-30", 999, 0),
    "C015": ("2026-10-02", 950, 0),     "C016": ("2026-10-06", 9200, 0),
}

def rules_only_decision(claim_id, receipt_date, receipt_total, non_reimbursable):
    p = json.loads(ec.check_policy_rules.run(claim_id=claim_id, receipt_total=receipt_total,
                                             non_reimbursable_amount=non_reimbursable))
    f = json.loads(ec.run_fraud_checks.run(claim_id=claim_id, receipt_date=receipt_date,
                                           receipt_total=receipt_total))
    triggers = p["rule_triggers"] + f["rule_triggers"]
    decision = ("REJECT" if any(t["effect"] == "REJECT" for t in triggers)
                else "FLAG" if triggers else "APPROVE")
    amount = 0 if decision == "REJECT" else p["recommended_amount"]
    return decision, amount, "; ".join(t["rule"] for t in triggers) or "none"

expected = pd.read_csv("data/expected_outcomes.csv")
rows = [(cid, *rules_only_decision(cid, *facts)) for cid, facts in perfect_receipts.items()]
baseline = pd.DataFrame(rows, columns=["claim_id", "decision", "amount", "rules_triggered"]).merge(expected, on="claim_id")
baseline["correct"] = (baseline.decision == baseline.expected_decision) & ((baseline.amount - baseline.expected_amount).abs() < 1)
print(f"Tools + rules match the answer key on {baseline.correct.sum()} of {len(baseline)} claims")
baseline[["claim_id", "decision", "expected_decision", "amount", "expected_amount", "rules_triggered", "correct"]]

## 5. Run the crew on one claim (watch the agents work)

Claim **C004** is a good demo: a client dinner on a Saturday where the bill includes four beers. The Receipt Analyst must spot the alcohol, the Policy Expert must remove it, the Fraud Auditor must notice the weekend, and the Finance Approver must combine everything.

`verbose=True` prints each agent's reasoning and tool calls. The crew uses Gemini Flash-Lite first because it is busy (error 503) less often on the free tier. If it is still busy, the code waits, retries, and then switches to Gemini Flash automatically. Red "503" messages during the run are normal and do not mean the run failed.

In [ ]:
demo = ec.run_claim("C004", verbose=True)

In [ ]:
def show(result):
    if "error" in result:
        print("Failed:", result["error"]); return
    d = result["decision"]
    print(f"Claim {result['claim_id']}  |  model: {result['model']}  |  {result['seconds']} s")
    print(f"DECISION: {d['decision']}   recommended amount: INR {d['recommended_amount']:,.0f}")
    print("Reasons:")
    for r in d["reasons"]:
        print("  -", r)
    print("Message to employee:", d["employee_message"])
    print("Receipt facts:", json.dumps(result["receipt"], indent=1))

show(demo)

## 6. Run the crew on all 16 claims

Each claim needs several Gemini calls, and the free tier allows only a limited number of requests per minute and per day. So results are **saved after every claim** to `results/crew_results.json`, and claims already done are skipped. If the run stops (quota or network), just run the cell again later and it continues where it stopped.

The free tier allows only about 15 Gemini requests per minute, so the crew paces itself at 10 per minute and waits whenever Google asks it to. Messages like "Gemini busy or rate-limited, waiting 25 s" are normal. Expect about 1 to 2 minutes per claim, so roughly 20 to 30 minutes for all 16.

In [ ]:
RESULTS_FILE = "results/crew_results.json"
results = json.load(open(RESULTS_FILE)) if os.path.exists(RESULTS_FILE) else {}

for cid in ec.CLAIMS.claim_id:
    if cid in results and "error" not in results[cid]:
        continue
    print(f"Running {cid} ...", end=" ", flush=True)
    results[cid] = ec.run_claim(cid)
    r = results[cid]
    print(r["decision"]["decision"] if "error" not in r else "ERROR: " + r["error"][:120])
    json.dump(results, open(RESULTS_FILE, "w"), indent=1)
    if "error" in r and r["error"].startswith("DAILY QUOTA"):
        print()
        print("The free daily Gemini quota is used up. Finished claims are saved;")
        print("run this cell again tomorrow (or with a key from another Google project) to continue.")
        break

print(f"Done: {sum('error' not in r for r in results.values())} of {len(ec.CLAIMS)} claims have results")

## 7. Evaluation: how often do the agents get it right?

We compare the crew's decision and recommended amount with the answer key.

In [ ]:
rows = []
for cid, r in results.items():
    if "error" in r:
        continue
    d = r["decision"]
    rows.append({"claim_id": cid, "crew_decision": d["decision"], "crew_amount": d["recommended_amount"],
                 "receipt_total_read": r["receipt"].get("receipt_total"),
                 "non_reimbursable_found": sum(i["amount"] for i in r["receipt"].get("non_reimbursable_items", [])),
                 "seconds": r["seconds"], "model": r["model"], "reasons": " | ".join(d["reasons"])})

evaluation = pd.DataFrame(rows).merge(expected, on="claim_id")
evaluation["decision_correct"] = evaluation.crew_decision == evaluation.expected_decision
evaluation["amount_correct"] = (evaluation.crew_amount - evaluation.expected_amount).abs() < 1

n = len(evaluation)
print(f"Claims evaluated: {n}")
print(f"Decision accuracy: {evaluation.decision_correct.mean():.0%} ({evaluation.decision_correct.sum()}/{n})")
print(f"Amount accuracy:   {evaluation.amount_correct.mean():.0%} ({evaluation.amount_correct.sum()}/{n})")
print(f"Average time per claim: {evaluation.seconds.mean():.0f} s")

evaluation.to_csv("results/evaluation.csv", index=False)
evaluation[["claim_id", "expected_decision", "crew_decision", "expected_amount", "crew_amount", "decision_correct", "amount_correct"]]

In [ ]:
# Where did the crew agree or disagree with the answer key?
pd.crosstab(evaluation.expected_decision, evaluation.crew_decision,
            rownames=["Expected"], colnames=["Crew decided"], margins=True)

In [ ]:
# Every claim the crew got wrong, with its reasoning (the most useful part for the report)
wrong = evaluation[~(evaluation.decision_correct & evaluation.amount_correct)]
print(f"{len(wrong)} claim(s) to review")
wrong[["claim_id", "expected_decision", "crew_decision", "expected_amount", "crew_amount", "reason", "reasons"]]

## 8. What to note for the report

- **Accuracy vs the rules-only baseline (Section 4):** the baseline is perfect only because it was given perfect receipt facts. The crew has to read the receipts itself, so any gap shows where reading or reasoning failed.
- **Mistakes in Section 7:** check whether each came from the Receipt Analyst misreading a receipt, an agent not calling its tool, or the Finance Approver applying the decision rules wrongly. Screenshot one or two as examples.
- **Cost and speed:** seconds per claim and number of Gemini calls; compare with a few minutes per claim for a human reviewer.

**Next step:** a Streamlit front end where an employee or finance user picks or enters a claim and watches the four agents work.